# Bézier Curves and the De Casteljau Algorithm

A **Bézier curve** of degree $n$ is a parametric polynomial curve defined by $n+1$ **control points** $P_0, P_1, \ldots, P_n \in \mathbb{R}^d$:
$$
\mathbf{B}(t) = \sum_{i=0}^{n} B_{i,n}(t)\, P_i, \qquad t \in [0, 1],
$$
where $B_{i,n}(t) = \binom{n}{i} t^i (1-t)^{n-i}$ are the **Bernstein basis polynomials** of degree $n$. They form a partition of unity ($\sum_i B_{i,n} = 1$) and are non-negative on $[0,1]$, so the curve always lies in the **convex hull** of its control polygon.

The **De Casteljau algorithm** (Paul de Casteljau, ~1959, independently Pierre Bézier) computes $\mathbf{B}(t)$ by successive **affine interpolation** without forming the Bernstein polynomials explicitly. Starting from $P_i^{(0)} = P_i$, the triangular scheme is:
$$
P_i^{(r)}(t) = (1-t)\,P_i^{(r-1)}(t) + t\,P_{i+1}^{(r-1)}(t), \qquad r = 1,\ldots,n, \quad i = 0,\ldots,n-r.
$$
The final value $P_0^{(n)}(t)$ is the point on the curve at parameter $t$.

**Key properties of Bézier curves:**
- **Interpolation of endpoints**: $\mathbf{B}(0) = P_0$, $\mathbf{B}(1) = P_n$.
- **Tangent at endpoints**: $\mathbf{B}'(0) = n(P_1 - P_0)$, $\mathbf{B}'(1) = n(P_n - P_{n-1})$.
- **Convex hull property**: $\mathbf{B}(t) \in \mathrm{conv}(P_0,\ldots,P_n)$ for all $t$.
- **Affine invariance**: applying an affine map to the control points maps the curve.
- **Subdivision**: the De Casteljau intermediate points define two sub-curves over $[0,t]$ and $[t,1]$.

Bézier curves are the backbone of font design, computer-aided design (CAD), SVG graphics, and animation systems.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["figure.dpi"] = 120

## The De Casteljau triangular scheme

For a fixed parameter $t_0$, the De Casteljau scheme computes a triangular array of points. The intermediate levels $P_i^{(r)}$ shrink by one row at each level until a single point remains — the point on the curve at $t = t_0$. All intermediate points lie on the curve's subdivisions, making the algorithm numerically stable and geometrically intuitive.

In [ ]:
def de_casteljau(P, t):
    """Compute Bezier point at t given control points P (shape (n+1, d)).
    Returns the full triangular array (list of arrays at each level)."""
    P = np.asarray(P, dtype=float)
    levels = [P.copy()]
    Q = P.copy()
    while len(Q) > 1:
        Q = (1 - t) * Q[:-1] + t * Q[1:]
        levels.append(Q.copy())
    return levels


def bezier_curve(P, n_pts=300):
    """Evaluate Bezier curve at n_pts parameter values in [0,1]."""
    P = np.asarray(P, dtype=float)
    ts = np.linspace(0, 1, n_pts)
    curve = np.array([de_casteljau(P, t)[-1][0] for t in ts])
    return curve


# Example control polygon
P_demo = np.array([[0.1, 0.1],
                   [0.2, 0.9],
                   [0.5, 0.6],
                   [0.7, 0.95],
                   [0.9, 0.2]])

curve_demo = bezier_curve(P_demo)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*P_demo.T, "k--o", lw=1.2, ms=9, label="control polygon", zorder=3)
ax.plot(*curve_demo.T, "royalblue", lw=2.5, label="Bézier curve", zorder=2)
for i, p in enumerate(P_demo):
    ax.text(p[0] + 0.02, p[1] + 0.02, f"$P_{i}$", fontsize=11)
ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.set_title("Degree-4 Bézier curve")
ax.legend(); ax.axis("off")
plt.tight_layout()
plt.show()

## De Casteljau construction at a fixed $t$

For $t = t_0$, the intermediate points of the scheme are drawn as nested sub-polygons that converge to the point on the curve. Each level $r$ connects points $P_i^{(r)}$ with a line of the same hue as the progress. The red star marks $\mathbf{B}(t_0)$.

In [ ]:
def show_construction(t0=0.4):
    levels = de_casteljau(P_demo, t0)
    n_lev = len(levels)
    curve = bezier_curve(P_demo)

    fig, ax = plt.subplots(figsize=(6, 6))
    # full curve (faint)
    ax.plot(*curve.T, color="royalblue", lw=2, alpha=0.35, zorder=1)
    # control polygon
    ax.plot(*P_demo.T, "k--o", lw=1.2, ms=9, zorder=3, label="control polygon")
    # intermediate levels
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, n_lev - 1))
    for r in range(1, n_lev):
        pts = levels[r]
        ax.plot(*pts.T, "-o", lw=1.8, ms=7, color=colors[r - 1],
                label=f"level $r={r}$", zorder=r + 3)
    # final point
    ax.plot(*levels[-1][0], "r*", ms=15, zorder=20,
            label=f"$\\mathbf{{B}}({t0:.2f})$")
    ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_title(f"De Casteljau construction at $t = {t0:.2f}$")
    ax.legend(fontsize=8, loc="lower right"); ax.axis("off")
    plt.tight_layout(); plt.show()


show_construction(0.4)

## Interactive construction slider

Drag the parameter $t$ to watch the De Casteljau point trace the Bézier curve.

In [ ]:
interact(show_construction,
         t0=FloatSlider(value=0.4, min=0.0, max=1.0, step=0.01,
                        description="$t$"));

## Bernstein basis polynomials

The Bernstein polynomials $B_{i,n}(t) = \binom{n}{i}t^i(1-t)^{n-i}$ weight the control points in the Bézier representation. Key properties:
- They partition unity: $\sum_{i=0}^n B_{i,n}(t) = 1$ for all $t$.
- They are all non-negative on $[0,1]$.
- $B_{i,n}$ peaks near $t = i/n$, so control point $P_i$ has its largest influence close to that parameter value.

In [ ]:
from math import comb

t_arr = np.linspace(0, 1, 400)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, deg in zip(axes, [3, 5, 8]):
    for i in range(deg + 1):
        s = i / deg
        B = comb(deg, i) * t_arr**i * (1 - t_arr)**(deg - i)
        ax.plot(t_arr, B, lw=2, color=(s, 0, 1 - s),
                label=f"$B_{{{i},{deg}}}$")
    ax.set_title(f"Degree $n = {deg}$")
    ax.set_xlabel("$t$")
    ax.set_ylabel("$B_{i,n}(t)$" if ax is axes[0] else "")
    ax.legend(fontsize=7, ncol=2)
fig.suptitle("Bernstein basis polynomials", y=1.02)
plt.tight_layout()
plt.show()

## Subdivision property

De Casteljau at $t = t_0$ splits the curve into **two sub-curves** of the same degree, with control points given by the two edges of the triangular array:
- Left sub-curve on $[0, t_0]$: control points $P_0^{(0)}, P_0^{(1)}, \ldots, P_0^{(n)}$.
- Right sub-curve on $[t_0, 1]$: control points $P_0^{(n)}, P_1^{(n-1)}, \ldots, P_n^{(0)}$.

This subdivision is the key to adaptive rendering, boolean operations, and computation of intersections in CAD systems.

In [ ]:
def subdivide(P, t0):
    """Return control points of left and right sub-curves."""
    levels = de_casteljau(P, t0)
    left  = np.array([lev[0] for lev in levels])   # first point at each level
    right = np.array([lev[-1] for lev in levels[::-1]])  # last point, reversed
    return left, right


t_split = 0.5
P_left, P_right = subdivide(P_demo, t_split)

c_left  = bezier_curve(P_left)
c_right = bezier_curve(P_right)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (sub, col, title) in zip(axes, [
    (c_left,  "crimson",   "Left sub-curve $[0, 0.5]$"),
    (c_right, "royalblue", "Right sub-curve $[0.5, 1]$"),
]):
    ax.plot(*bezier_curve(P_demo).T, "gray", lw=1.5, ls="--", alpha=0.4,
            label="original")
    ax.plot(*P_demo.T, "k--o", lw=1, ms=6, alpha=0.3)
    ax.plot(*sub.T, col, lw=2.5, label=title)
    if sub is c_left:
        ax.plot(*P_left.T, "o--", color=col, lw=1.2, ms=7, label="sub-polygon")
    else:
        ax.plot(*P_right.T, "o--", color=col, lw=1.2, ms=7, label="sub-polygon")
    ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_title(title); ax.legend(fontsize=9); ax.axis("off")
fig.suptitle(f"De Casteljau subdivision at $t = {t_split}$", y=1.02)
plt.tight_layout()
plt.show()

## Degree comparison

Higher-degree Bézier curves can represent more complex shapes but oscillate more (Runge phenomenon for polynomial interpolation). In practice, smooth shapes are built as **piecewise** Bézier curves (splines) with continuity conditions at the joints.

In [ ]:
rng = np.random.default_rng(99)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))
for ax, n_deg in zip(axes, [2, 4, 6, 9]):
    # fixed random control points on a circle-ish arrangement
    angles = np.linspace(0, 2 * np.pi, n_deg + 1, endpoint=False)
    P = np.column_stack([
        0.5 + 0.35 * np.cos(angles) + rng.uniform(-0.08, 0.08, n_deg + 1),
        0.5 + 0.35 * np.sin(angles) + rng.uniform(-0.08, 0.08, n_deg + 1),
    ])
    curve = bezier_curve(P)
    ax.plot(*P.T, "k--o", lw=1, ms=6, alpha=0.5)
    ax.plot(*curve.T, lw=2.5, color=(n_deg / 9, 0, 1 - n_deg / 9))
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"degree {n_deg}")
fig.suptitle("Bézier curves of increasing degree", y=1.02)
plt.tight_layout()
plt.show()

## Bibliographical resources

- de Casteljau, P. (1959). Outillages méthodes calcul. Internal report, Citroën.
- Bézier, P. (1972). *Numerical Control: Mathematics and Applications*. Wiley.
- Farin, G. (2001). *Curves and Surfaces for CAGD: A Practical Guide* (5th ed.). Morgan Kaufmann.
- Hoschek, J. and Lasser, D. (1993). *Fundamentals of Computer Aided Geometric Design*. A K Peters.
- Prautzsch, H., Boehm, W., and Paluszny, M. (2002). *Bézier and B-Spline Techniques*. Springer.